## Recieve and Process Messages from Service Bus Queue

### Installing Libraries and Utilities

In [ ]:
%pip install azure-servicebus==7.14.3 openai==2.38.0 python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading service bus configurations
service_bus_namespace = os.getenv("SERVICE_BUS_NAMESPACE")
service_bus_connection_string = os.getenv("SERVICE_BUS_CONNECTION_STRING")
service_bus_queue_name = os.getenv("SERVICE_BUS_QUEUE_NAME")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Service Bus Client

In [ ]:
from azure.servicebus import ServiceBusClient
from azure.core.credentials import AzureKeyCredential

sb_client = ServiceBusClient.from_connection_string(
    conn_str = service_bus_connection_string
)

### Helper Function to Process User Queries

In [ ]:
from openai import AzureOpenAI

def process_user_message(user_query, llm):
    azure_openai_client = AzureOpenAI(
        azure_endpoint = azure_openai_endpoint,
        api_version = "2024-06-01",
        api_key = azure_openai_api_key
    )

    response = azure_openai_client.chat.completions.create(
        model = llm,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI assistant"
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        temperature = 0.7
    )

    return response.choices[0].message.content

### Process Messages in Peek-Lock Mode Reliably

In [ ]:
from azure.servicebus import ServiceBusReceiveMode
import json

# create a queue receiver object to reliably process messages
receiver = sb_client.get_queue_receiver(
    queue_name = service_bus_queue_name,
    receive_mode = ServiceBusReceiveMode.PEEK_LOCK,
    max_wait_time = 60
)

for msg in receiver:
    try:
        payload = json.loads(str(msg))
        print("user query: {}".format(payload.get("prompt")))
        print("\n")
        assistant_response = process_user_message(payload.get("prompt"), payload.get("model"))
        print("Assistant Reponse: {}".format(assistant_response))
        print("============================================")

        # Mark the message as complete after successful execution
        receiver.complete_message(msg)
    except Exception as e:
        receiver.dead_letter_message(
            msg,
            reason = "Model deployment does not exist",
            error_description = "Model deployment does not exist"
        )
